### **Submissão 2B — Modelo PyTorch**

**Grupo 1 · MIA · Aprendizagem Profunda**

Modelo: DNN from Pytorch
Output: `subm2-g1-MIA-B.csv`

In [ ]:
import numpy as np
import pandas as pd
import pickle
import sys, os, re
import torch
import torch.nn as nn
from collections import Counter

sys.path.append(os.path.abspath('../src'))
from utils import transform_new_texts

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

# 1. Carregar metadados
print('1. A carregar modelo PyTorch...')
with open('../Subm2/pytorch.pkl', 'rb') as f:
    meta = pickle.load(f)

model_class = meta['model_class']
input_size = meta['input_size']
num_classes = meta['num_classes']
class_names = meta['class_names']
transformers = meta['transformers']
encoder = meta['encoder']

print(f'   Modelo: {model_class}')
print(f'   Classes: {class_names}')

Dispositivo: cuda
1. A carregar modelo PyTorch...
   Modelo: DNNGrid
   Classes: [np.str_('Anthropic'), np.str_('Google'), np.str_('Human'), np.str_('Meta'), np.str_('OpenAI')]


In [12]:
# 2. Carregar dataset
print('2. A carregar dataset...')
df = pd.read_csv('../database/dataset-subm2.csv', sep=';')
df.columns = df.columns.str.strip().str.lower()
textos = df['text'].tolist()
ids = df['id'].tolist()
print(f'   {len(textos)} textos carregados')

2. A carregar dataset...
   150 textos carregados


In [13]:
# 3. Definir todos os modelos
print('3. A preparar modelo...')

class DNNWide(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes))
    def forward(self, x): return self.net(x)

class DNNNarrow(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, num_classes))
    def forward(self, x): return self.net(x)

class DNNDeep(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes))
    def forward(self, x): return self.net(x)

class DNNLeakyReLU(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.1), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.Dropout(0.4),
            nn.Linear(128, num_classes))
    def forward(self, x): return self.net(x)

class DNNELU(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ELU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ELU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes))
    def forward(self, x): return self.net(x)

class DNNSimple(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 32), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(32, num_classes))
    def forward(self, x): return self.net(x)

class DNNVeryDeep(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, num_classes))
    def forward(self, x): return self.net(x)

class DNNGrid(nn.Module):
    def __init__(self, input_size, num_classes, drop=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.1), nn.Dropout(drop),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.Dropout(drop),
            nn.Linear(128, num_classes))
    def forward(self, x): return self.net(x)

class EmbeddingDNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes))
    def forward(self, x): return self.fc(self.embedding(x).mean(dim=1))

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim*2, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes))
    def forward(self, x):
        out, _ = self.lstm(self.embedding(x))
        return self.fc(out[:, -1, :])

class BiGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim*2, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes))
    def forward(self, x):
        out, _ = self.gru(self.embedding(x))
        return self.fc(out[:, -1, :])

class BiLSTMGloVe(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_classes, freeze_embed=True):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(torch.FloatTensor(embedding_matrix))
        if freeze_embed: self.embedding.weight.requires_grad = False
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim*2, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes))
    def forward(self, x):
        out, _ = self.lstm(self.embedding(x))
        return self.fc(out[:, -1, :])

3. A preparar modelo...


In [ ]:
# 4. Reconstruir e classificar
print('4. A classificar...')

MAX_SEQ_LEN = 150

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return [t for t in text.split() if len(t) > 1]

def texts_to_sequences(texts, word2idx, max_len):
    seqs = []
    for text in texts:
        tokens = simple_tokenize(text)
        seq = [word2idx.get(t, 1) for t in tokens[:max_len]]
        seqs.append(seq + [0] * (max_len - len(seq)))
    return np.array(seqs)

tab_map = {
    'DNNWide': DNNWide, 'DNNNarrow': DNNNarrow, 'DNNDeep': DNNDeep,
    'DNNLeakyReLU': DNNLeakyReLU, 'DNNELU': DNNELU,
    'DNNSimple': DNNSimple, 'DNNVeryDeep': DNNVeryDeep,
}

if model_class == 'DNNGrid':
    grid_drop = meta.get('grid_drop', 0.5)
    model = DNNGrid(input_size, num_classes, drop=grid_drop)
    is_sequential = False
elif model_class in tab_map:
    model = tab_map[model_class](input_size, num_classes)
    is_sequential = False
elif model_class == 'EmbeddingDNN':
    model = EmbeddingDNN(meta['vocab_size'], meta['embed_dim'], num_classes)
    is_sequential = True
elif model_class == 'BiLSTM':
    model = BiLSTM(meta['vocab_size'], meta['embed_dim'], meta['hidden_dim'], num_classes)
    is_sequential = True
elif model_class == 'BiGRU':
    model = BiGRU(meta['vocab_size'], meta['embed_dim'], meta['hidden_dim'], num_classes)
    is_sequential = True
elif model_class == 'BiLSTMGloVe':
    model = BiLSTMGloVe(meta['embedding_matrix'], meta['hidden_dim'], num_classes,
                         freeze_embed=meta.get('freeze_embed', True))
    is_sequential = True
else:
    raise ValueError(f'Modelo desconhecido: {model_class}')

model.load_state_dict(torch.load('../Subm2/pytorch.pth', map_location=device, weights_only=True))
model = model.to(device)
model.eval()
print(f'   Modelo {model_class} carregado')

if is_sequential:
    word2idx = meta['word2idx']
    max_seq_len = meta.get('max_seq_len', MAX_SEQ_LEN)
    seqs = texts_to_sequences(textos, word2idx, max_seq_len)
    X = torch.LongTensor(seqs).to(device)
else:
    X_np = transform_new_texts(textos, transformers)
    X = torch.FloatTensor(X_np).to(device)

with torch.no_grad():
    out = model(X)
    preds = torch.argmax(out, 1).cpu().numpy()

labels_pred = [class_names[i] for i in preds]
print(f'   ✅ {len(labels_pred)} previsões feitas')

4. A classificar...
   Modelo DNNGrid carregado
   ✅ 150 previsões feitas


In [15]:
# 5. Exportar CSV
print('5. A exportar...')

df_out = pd.DataFrame({'ID': ids, 'Label': labels_pred})

os.makedirs('../Subm2', exist_ok=True)
output_path = '../Subm2/subm2-g1-MIA-B.csv'
df_out.to_csv(output_path, sep=';', index=False, encoding='utf-8')

print(f'✅ {output_path}')
print(f'\nDistribuição:')
print(df_out['Label'].value_counts().to_string())

assert len(df_out) == 150
assert list(df_out.columns) == ['ID', 'Label']
print(f'\n✅ OK: {len(df_out)} linhas')

5. A exportar...
✅ ../Subm2/subm2-g1-MIA-B.csv

Distribuição:
Label
Human        59
Meta         27
OpenAI       25
Google       24
Anthropic    15

✅ OK: 150 linhas
